# MLP playground: tune it and see what changes

<a href="https://colab.research.google.com/github/buildwithyang/net-attack-predict/blob/main/notebooks/playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IoT network attack detection (RT-IoT2022) with the project's own code in `src/iotids/`.
Change the data steps, the network, the optimizer and the training settings with sliders,
press **Train**, and watch the loss curves move while it trains.

The notebook does not have its own model or training loop. It calls the same functions as
`scripts/train.py`: `load_splits → build_model → fit → choose_threshold → evaluate`.
Same settings + same seed = the same numbers as the command line.

**How to open it**

| Where | How |
| --- | --- |
| Colab (easiest) | Click the badge above, or in Colab: *File → Open notebook → GitHub* → `buildwithyang/net-attack-predict` → `notebooks/playground.ipynb`. Then *Runtime → Run all*. |
| Colab from Google Drive | Double-click the file in Drive → *Open with Google Colaboratory*. The setup cell downloads the project code from GitHub. |
| Local Jupyter / VS Code | Open the file inside the project folder and run all cells. |

A CPU is enough. One baseline run takes about 10 s on a laptop, a bit longer on Colab.

## 1. Setup

Finds the project (or downloads it from GitHub in Colab), unzips the dataset once, and makes
`import iotids` work. Nothing to change here.

In [ ]:
# Setup: find (or download) the project, unzip the data, make `import iotids` work.
import os, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = "https://github.com/buildwithyang/net-attack-predict.git"
IN_COLAB = "google.colab" in sys.modules


def find_project_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "src" / "iotids").is_dir():
            return folder
    return None


ROOT = find_project_root()
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from inside the project folder (notebooks/), or use Colab.")
    ROOT = Path("/content/net-attack-predict")
    if not ROOT.exists():  # to get newer code later: !git -C /content/net-attack-predict pull
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)  # config paths such as data/raw/RT_IOT2022.csv are relative to the project root
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # import iotids straight from src/, no pip install needed

csv_path = ROOT / "data" / "raw" / "RT_IOT2022.csv"
if not csv_path.exists():
    with zipfile.ZipFile(ROOT / "data" / "raw" / "rt-iot2022.zip") as z:
        z.extractall(csv_path.parent)

try:
    import ipywidgets  # noqa: F401  (Colab already has it)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"], check=True)

print("project:", ROOT)
print("data:   ", csv_path, f"({csv_path.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# Project code + notebook helpers (run once).
import copy, io, json, tempfile, time, traceback

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
import pandas as pd
import yaml
from IPython.display import HTML, Image, clear_output, display

from iotids import plots
from iotids.data import load_splits
from iotids.decision import choose_threshold
from iotids.evaluate import evaluate
from iotids.model import ACTIVATIONS, build_model
from iotids.train import OPTIMIZERS, WITH_MOMENTUM, fit, predict_proba

BASELINE = yaml.safe_load(open("configs/baseline.yaml"))  # every run is described as "baseline + changes"
RUNS = {}  # name -> results of one run; lives as long as this notebook session


def show(fig, width=980):
    # Draw a figure as a PNG. Works the same in Colab, Jupyter and VS Code.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    display(Image(buf.getvalue(), width=width))


_split_cache = {}


def get_splits(data_cfg, seed):
    # load_splits reads the 55 MB CSV (a few seconds), so keep the last 3 results in memory.
    key = json.dumps([data_cfg, seed], sort_keys=True, default=str)
    if key not in _split_cache:
        if len(_split_cache) >= 3:
            _split_cache.pop(next(iter(_split_cache)))
        _split_cache[key] = load_splits(data_cfg, seed)
    return _split_cache[key]


def _short(value):
    if isinstance(value, dict):
        params = ", ".join(f"{k}={v}" for k, v in value.items() if k != "name")
        return f"{value['name']}({params})" if params else value["name"]
    return str(value)


def changes(cfg, base=BASELINE):
    # What differs from configs/baseline.yaml, e.g. ['lr=0.1', 'hidden=[8]'].
    out = [] if cfg["seed"] == base["seed"] else [f"seed={cfg['seed']}"]
    for section in ("data", "model", "train"):
        new, old = cfg[section], base[section]
        for key in dict.fromkeys([*old, *new]):
            if new.get(key) != old.get(key):
                out.append(f"{key}={_short(new.get(key))}")
    if cfg["decision"] != base["decision"]:
        d = cfg["decision"]
        out.append(f"threshold={d['value']}" if d["name"] == "fixed" else f"max_fpr={d['fpr']}")
    return out


def auto_name(cfg):
    return unique_name(", ".join(changes(cfg)) or "baseline")


def unique_name(name):
    unique, n = name, 2
    while unique in RUNS:
        unique, n = f"{name} #{n}", n + 1
    return unique


class LivePlot:
    # on_epoch callback for fit(): redraws the curves while training runs.
    # It only reads the model (val predictions); it never changes it.

    def __init__(self, out, splits, cfg, min_seconds=0.7):
        self.out, self.val, self.cfg = out, splits.val, cfg
        self.min_seconds, self.last_draw, self.start = min_seconds, 0.0, time.time()
        self.false_alarm, self.missed = [], []

    def __call__(self, epoch, history, model):
        flagged = predict_proba(model, self.val.X) >= 0.5
        attack = self.val.y == 1
        self.false_alarm.append(100 * (flagged & ~attack).sum() / max(1, (~attack).sum()))
        self.missed.append(100 * (~flagged & attack).sum() / max(1, attack.sum()))
        if time.time() - self.last_draw >= self.min_seconds:
            self.draw(epoch, history)

    def draw(self, epoch, history, done=False):
        self.last_draw = time.time()
        total = self.cfg["train"]["epochs"]
        per_epoch = (time.time() - self.start) / max(1, epoch)
        fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
        epochs = np.arange(1, len(history.train_loss) + 1)
        left.plot(epochs, _finite(history.train_loss), color=plots.TRAIN_LINE, lw=2, ls="--", label="train loss")
        left.plot(epochs, _finite(history.val_loss), color=plots.VAL_LINE, lw=2, label="validation loss")
        if history.best_epoch:
            left.axvline(history.best_epoch, color=plots.TEXT_2, lw=1, ls=":")
            left.annotate(f"best val: epoch {history.best_epoch}", (history.best_epoch, 1),
                          xycoords=("data", "axes fraction"), xytext=(4, -6), textcoords="offset points",
                          fontsize=10, color=plots.TEXT_2, va="top")
        if np.isfinite(history.val_loss).any() and np.nanmin(_finite(history.val_loss)) > 0:
            left.set_yscale("log")
        left.set_xlim(0.5, max(total, 2) + 0.5)
        left.set_title("Loss per epoch (lower is better)")
        left.set_xlabel("epoch")
        left.legend(loc="lower left")
        right.plot(epochs, self.false_alarm, color=plots.NORMAL, lw=2, label="false alarms (normal flagged)")
        right.plot(epochs, self.missed, color=plots.ATTACK, lw=2, label="missed attacks")
        right.set_xlim(0.5, max(total, 2) + 0.5)
        right.set_ylim(bottom=0)
        right.set_title("Validation set at threshold 0.5 (%, lower is better)")
        right.set_xlabel("epoch")
        right.legend(loc="best")
        for ax in (left, right):
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        state = "finished" if done else "training"
        fig.suptitle(f"{state}: epoch {epoch}/{total}  ·  {per_epoch:.2f} s per epoch", x=0.01, ha="left",
                     fontsize=13, fontweight="bold", color=plots.TEXT)
        fig.tight_layout()
        with self.out:
            clear_output(wait=True)
            show(fig)


def _finite(values):
    a = np.asarray(values, dtype=float)
    return np.where(np.isfinite(a), a, np.nan)


def run(cfg, live_out=None):
    # The same steps as iotids.experiment.run_experiment, but results stay in memory (no files).
    cfg = copy.deepcopy(cfg)
    seed = int(cfg["seed"])
    start = time.time()
    splits = get_splits(cfg["data"], seed)                               # 1. data
    model = build_model(cfg["model"], splits.n_features, seed)           # 2. model
    live = LivePlot(live_out, splits, cfg) if live_out is not None else None
    history = fit(model, splits.train, splits.val, cfg["train"], seed, on_epoch=live)  # 3. train
    if live:
        live.draw(len(history.train_loss), history, done=True)
    val_scores = predict_proba(model, splits.val.X)
    threshold = choose_threshold(val_scores, splits.val.y, cfg["decision"])  # 4. threshold, on val only
    test_scores = predict_proba(model, splits.test.X)                    # 5. test, once
    report = evaluate(test_scores, splits.test, threshold)               # 6. numbers
    result = {
        "name": cfg["name"], "cfg": cfg, "history": history, "report": report,
        "test_scores": test_scores, "test": splits.test, "fingerprint": splits.test_fingerprint,
        "n_train": len(splits.train.y), "n_features": splits.n_features,
        "seconds": time.time() - start, "diverged": not np.isfinite(test_scores).all(),
    }
    RUNS[cfg["name"]] = result
    return result


def show_summary(result):
    # The same four charts as runs/<name>/summary.png.
    report, history = result["report"], result["history"]
    if result["diverged"]:
        display(HTML("<b style='color:#c0392b'>Training diverged: the loss became NaN/inf, so the scores are "
                     "meaningless. Try a smaller learning rate or turn scaling on.</b>"))
        return
    fig, axes = plt.subplots(2, 2, figsize=(16, 11.5), gridspec_kw={"width_ratios": [1, 1.25]})
    plots.plot_training_curves(axes[0, 0], history)
    plots.plot_per_type(axes[0, 1], report)
    plots.plot_confusion_matrix(axes[1, 0], report)
    plots.plot_score_distribution(axes[1, 1], result["test_scores"], result["test"].y, report.threshold)
    fig.suptitle(f"Test set results: {result['name']}", x=0.01, ha="left", fontsize=17, fontweight="bold",
                 color=plots.TEXT)
    fig.text(0.01, 0.935, plots.headline(report), ha="left", fontsize=13, color=plots.TEXT_2)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    show(fig)


def results_table():
    rows = []
    for name, r in RUNS.items():
        rep, h = r["report"], r["history"]
        val = _finite(h.val_loss)
        rows.append({
            "run": name,
            "train rows": r["n_train"],
            "epochs run": len(h.train_loss),
            "kept epoch": h.kept_epoch,
            "best val loss": np.nanmin(val) if np.isfinite(val).any() else np.nan,
            "attacks detected %": 100 * rep.detection_rate,
            "false alarms %": 100 * rep.false_alarm_rate,
            "accuracy %": 100 * rep.accuracy,
            "F1": rep.f1,
            "seconds": r["seconds"],
        })
    table = pd.DataFrame(rows).set_index("run") if rows else pd.DataFrame()
    return table.round({"best val loss": 4, "attacks detected %": 2, "false alarms %": 2, "accuracy %": 2,
                        "F1": 4, "seconds": 1})


def _clip(text, n=34):
    return text if len(text) <= n else text[: n - 1] + "…"


def compare(names, title="Comparison"):
    # The same chart as scripts/compare.py: val loss curves + false alarms + missed attacks.
    runs = [RUNS[n] for n in names]
    if len({r["fingerprint"] for r in runs}) > 1:
        display(HTML("<b>Warning:</b> these runs use different test sets (different seed or split), "
                     "so their test numbers are not directly comparable."))
    # Chart labels: the first run is the reference; the others show only what differs from it.
    labels = [_clip(runs[0]["name"])] + [_clip(", ".join(changes(r["cfg"], runs[0]["cfg"])) or "same settings")
                                         for r in runs[1:]]
    if labels != names:
        display(pd.DataFrame({"run": names}, index=pd.Index(labels, name="label in the chart")))
    metrics = [{"name": label, "test": r["report"].to_dict()} for label, r in zip(labels, runs)]
    histories = [{"train_loss": list(_finite(r["history"].train_loss)),
                  "val_loss": list(_finite(r["history"].val_loss))} for r in runs]
    path = Path(tempfile.gettempdir()) / "iotids_compare.png"
    plots.save_comparison(metrics, histories, path, title)
    display(Image(filename=str(path), width=980))


print("ready. optimizers:", ", ".join(OPTIMIZERS), "| activations:", ", ".join(ACTIVATIONS))

## 2. The data at a glance

Loads the baseline data once (it is cached, so the first training run starts faster).
The model only ever sees the number matrix `X` and the labels `y` (0 = normal, 1 = attack).

In [ ]:
splits = get_splits(BASELINE["data"], BASELINE["seed"])
summary = pd.DataFrame({
    part: {"rows": len(s.y), "normal": int((s.y == 0).sum()), "attack": int(s.y.sum()),
           "attack share %": round(100 * s.y.mean(), 1)}
    for part, s in [("train", splits.train), ("val", splits.val), ("test", splits.test)]
}).T
display(summary)
print(f"X has {splits.n_features} columns (numeric features + one-hot proto/service).",
      f"Test set fingerprint: {splits.test_fingerprint}")

types, counts = np.unique(splits.train.attack_type, return_counts=True)
order = np.argsort(counts)
fig, ax = plt.subplots(figsize=(9, 4.8))
colors = [plots.ATTACK if y else plots.NORMAL for y in
          [splits.train.y[splits.train.attack_type == t][0] for t in types[order]]]
ax.barh(types[order], counts[order], color=colors)
ax.set_xscale("log")
ax.set_title("Training rows per traffic type (log scale; blue = normal, orange = attack)")
ax.set_xlabel("rows")
ax.grid(axis="y", visible=False)
show(fig, width=760)

About 90% of the rows are attacks, so "always say attack" already gets ~90% accuracy.
That is why the charts below focus on **false alarms** (normal traffic flagged as attack)
and **missed attacks**, not accuracy.

<details><summary><b>Optional: the training loop you are tuning</b> (click to open)</summary>

Every knob in the playground ends up in this function from `src/iotids/train.py`. Run the
next cell to print it. The four lines inside the batch loop are the whole of training:
forward → loss → backward → step.
</details>

In [ ]:
import inspect
import iotids.train
print(inspect.getsource(iotids.train.fit))

## 3. Playground

1. Pick a starting point (*Start from*: the baseline or any experiment in `configs/`).
2. Change settings in the tabs. Each run is named after what differs from the baseline.
3. Press **Train**. The live chart updates every epoch; when it finishes you get the test results
   and the run is added to the table (and to the *Compare* section below).

| Tab | Knob | What it changes | What to watch |
| --- | --- | --- | --- |
| Data | scaler | how features are rescaled (fitted on train only) | without scaling, features up to ~2e10 make training unstable |
| Data | training rows | use only part of the training set | fewer rows → overfits sooner (val loss turns up) |
| Model | hidden layers | e.g. `64` or `128, 64`; empty = no hidden layer (logistic regression) | capacity: too small underfits, too big can overfit |
| Model | activation, dropout | non-linearity; randomly switches off units while training | dropout makes train loss noisier but can lower val loss |
| Training | optimizer, learning rate | how each step uses the gradient, and how big the step is | too small = slow; too big = jumpy or stuck |
| Training | momentum | (sgd, rmsprop) keeps part of the last update | faster along consistent directions |
| Training | weight decay | pulls weights towards 0 (L2) | a regulariser, like dropout |
| Training | batch size | rows per gradient step | small = many noisy steps per epoch, slower per epoch |
| Training | early stopping | stop after N epochs without a better val loss, keep the best weights | 0 = off: keeps the last epoch, even if it overfits |
| Training | class weight | `balanced` makes each normal row count more in the loss | fewer false alarms, maybe more missed attacks |
| Decision | threshold rule | score ≥ threshold → "attack"; or pick it on val for a target false alarm rate | trades false alarms against missed attacks, no retraining |

To stop a long run: Colab *Runtime → Interrupt execution* (or the ■ button in Jupyter).

In [ ]:
CONFIG_FILES = sorted(str(p) for p in Path("configs").rglob("*.yaml"))
FEATURES = {"all": "all", "drop_constant": "drop_constant",
            "drop proto + service": {"name": "drop", "columns": ["proto", "service"]}}
SAMPLERS = {"all rows": "none",
            "50% (subsample)": {"name": "subsample", "fraction": 0.5},
            "10% (subsample)": {"name": "subsample", "fraction": 0.1},
            "5% (subsample)": {"name": "subsample", "fraction": 0.05},
            "1% (subsample)": {"name": "subsample", "fraction": 0.01},
            "undersample attacks to 1:1": {"name": "undersample", "ratio": 1.0}}
LRS = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0]
BATCHES = [16, 32, 64, 128, 256, 512, 1024, 4096]
DECAYS = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 0.1]
FPRS = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

S = {"description_width": "130px"}
L = W.Layout(width="440px")
w = {
    "start": W.Dropdown(options=CONFIG_FILES, value="configs/baseline.yaml", description="Start from", style=S, layout=L),
    "name": W.Text(placeholder="(empty = named after the changes)", description="Run name", style=S, layout=L),
    # data
    "scaler": W.Dropdown(options=["none", "standard", "minmax", "robust", "log_standard"], description="scaler", style=S, layout=L),
    "features": W.Dropdown(options=list(FEATURES), description="features", style=S, layout=L),
    "sampler": W.Dropdown(options=list(SAMPLERS), description="training rows", style=S, layout=L),
    # model
    "hidden": W.Text(description="hidden layers", placeholder="e.g. 64  or  128, 64", style=S, layout=L),
    "activation": W.Dropdown(options=list(ACTIVATIONS), description="activation", style=S, layout=L),
    "dropout": W.FloatSlider(min=0, max=0.7, step=0.05, description="dropout", style=S, layout=L),
    # training
    "optimizer": W.Dropdown(options=list(OPTIMIZERS), description="optimizer", style=S, layout=L),
    "lr": W.SelectionSlider(options=LRS, description="learning rate", style=S, layout=L),
    "momentum": W.FloatSlider(min=0, max=0.99, step=0.01, description="momentum", style=S, layout=L),
    "weight_decay": W.SelectionSlider(options=DECAYS, description="weight decay", style=S, layout=L),
    "batch_size": W.SelectionSlider(options=BATCHES, description="batch size", style=S, layout=L),
    "epochs": W.IntSlider(min=1, max=200, description="max epochs", style=S, layout=L),
    "early_stopping": W.IntSlider(min=0, max=20, description="early stopping", style=S, layout=L),
    "class_weight": W.Dropdown(options=["none", "balanced"], description="class weight", style=S, layout=L),
    "seed": W.IntText(description="seed", style=S, layout=W.Layout(width="250px")),
    # decision
    "rule": W.ToggleButtons(options=["fixed", "max_fpr"], description="threshold rule", style=S),
    "threshold": W.FloatSlider(min=0.05, max=0.95, step=0.05, description="threshold", style=S, layout=L),
    "fpr": W.SelectionSlider(options=FPRS, value=0.005, description="max false alarms", style=S, layout=L),
}


def parse_hidden(text):
    parts = text.replace(",", " ").split()
    if not all(p.isdigit() and int(p) >= 1 for p in parts):
        raise ValueError(f"hidden layers: use whole numbers >= 1, like 64 or 128, 64 (got {text!r})")
    return [int(p) for p in parts]


def current_cfg():
    cfg = copy.deepcopy(BASELINE)
    cfg["seed"] = int(w["seed"].value)
    cfg["data"].update(features=FEATURES[w["features"].value], scaler=w["scaler"].value,
                       sampler=SAMPLERS[w["sampler"].value])
    cfg["model"].update(hidden=parse_hidden(w["hidden"].value), activation=w["activation"].value,
                        dropout=round(w["dropout"].value, 2))
    t = cfg["train"]
    t.update(optimizer=w["optimizer"].value, lr=w["lr"].value, batch_size=w["batch_size"].value,
             epochs=w["epochs"].value, early_stopping=w["early_stopping"].value,
             class_weight=w["class_weight"].value)
    t.pop("momentum", None)
    t.pop("weight_decay", None)
    if w["optimizer"].value in WITH_MOMENTUM and w["momentum"].value > 0:
        t["momentum"] = round(w["momentum"].value, 2)
    if w["weight_decay"].value > 0:
        t["weight_decay"] = w["weight_decay"].value
    cfg["decision"] = ({"name": "fixed", "value": round(w["threshold"].value, 2)} if w["rule"].value == "fixed"
                       else {"name": "max_fpr", "fpr": w["fpr"].value})
    cfg["name"] = unique_name(w["name"].value.strip()) if w["name"].value.strip() else auto_name(cfg)
    return cfg


def _pick(options, value, what):
    # Nearest allowed slider value (config files may use values the sliders do not offer).
    if value in options:
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        best = min(options, key=lambda o: abs(np.log10(o + 1e-12) - np.log10(value + 1e-12)))
        print(f"note: {what}={value} is not on the slider, using {best}")
        return best
    raise ValueError(f"{what}={value!r} is not available in the playground")


def _key_for(mapping, value, what):
    for key, v in mapping.items():
        if v == value:
            return key
    raise ValueError(f"{what}={value!r} is not available in the playground")


def set_widgets(cfg):
    d, m, t, dec = cfg["data"], cfg["model"], cfg["train"], cfg["decision"]
    w["seed"].value = cfg["seed"]
    w["scaler"].value = d["scaler"]
    w["features"].value = _key_for(FEATURES, d["features"], "data.features")
    w["sampler"].value = _key_for(SAMPLERS, d["sampler"], "data.sampler")
    w["hidden"].value = ", ".join(str(h) for h in m["hidden"])
    w["activation"].value = m["activation"]
    w["dropout"].value = m["dropout"]
    w["optimizer"].value = t["optimizer"]
    w["lr"].value = _pick(LRS, t["lr"], "lr")
    w["momentum"].value = t.get("momentum", 0.0)
    w["weight_decay"].value = _pick(DECAYS, t.get("weight_decay", 0.0), "weight_decay") if t.get("weight_decay") else 0.0
    w["batch_size"].value = _pick(BATCHES, t["batch_size"], "batch_size")
    w["epochs"].value = t["epochs"]
    w["early_stopping"].value = t["early_stopping"]
    w["class_weight"].value = t["class_weight"] if t["class_weight"] in ("none", "balanced") else "balanced"
    w["rule"].value = dec["name"]
    if dec["name"] == "fixed":
        w["threshold"].value = dec["value"]
    else:
        w["fpr"].value = _pick(FPRS, dec["fpr"], "decision.fpr")
    w["name"].value = ""
    _refresh_enabled()


def _refresh_enabled(*_):
    w["momentum"].disabled = w["optimizer"].value not in WITH_MOMENTUM
    w["threshold"].disabled = w["rule"].value != "fixed"
    w["fpr"].disabled = w["rule"].value == "fixed"
    try:
        preview.value = f"<span style='color:#52514e'>next run will be called: <b>{current_cfg()['name']}</b></span>"
    except ValueError as err:
        preview.value = f"<span style='color:#c0392b'>{err}</span>"


train_btn = W.Button(description="Train", button_style="success", icon="play")
reset_btn = W.Button(description="Reset to baseline", icon="undo")
save_btn = W.Button(description="Save as config file", icon="save")
live_box = W.Checkbox(value=True, description="live chart", indent=False)
status, preview = W.HTML(), W.HTML()
live_out, result_out, table_out = W.Output(), W.Output(), W.Output()


def on_train(_):
    try:
        cfg = current_cfg()
    except ValueError as err:
        status.value = f"<span style='color:#c0392b'>{err}</span>"
        return
    train_btn.disabled = True
    status.value = f"training <b>{cfg['name']}</b> … (loading data the first time takes a few seconds)"
    result_out.clear_output()
    live_out.clear_output()
    try:
        result = run(cfg, live_out if live_box.value else None)
    except KeyboardInterrupt:
        status.value = "<b>stopped.</b> Nothing was saved for this run."
        return
    except Exception as err:  # show the problem in the panel (full traceback below it)
        status.value = f"<span style='color:#c0392b'><b>error:</b> {type(err).__name__}: {err}</span>"
        with result_out:
            traceback.print_exc()
        return
    finally:
        train_btn.disabled = False
    status.value = (f"done: <b>{result['name']}</b> in {result['seconds']:.1f} s "
                    f"({len(result['history'].train_loss)} epochs, kept epoch {result['history'].kept_epoch})")
    with result_out:
        show_summary(result)
    with table_out:
        clear_output(wait=True)
        display(results_table())
    refresh_run_lists()
    _refresh_enabled()


def on_save(_):
    cfg = current_cfg()
    name = w["name"].value.strip() or ", ".join(changes(cfg)) or "baseline"
    slug = "".join(c if c.isalnum() else "_" for c in name).strip("_").lower()[:60] or "run"
    path = Path("configs") / "notebook" / f"{slug}.yaml"
    path.parent.mkdir(parents=True, exist_ok=True)
    header = (f"# Saved from notebooks/playground.ipynb. Changes vs baseline: {', '.join(changes(cfg)) or 'none'}\n"
              f"# Run it from the project root: python scripts/train.py {path}\n")
    path.write_text(header + yaml.safe_dump({**cfg, "name": slug}, sort_keys=False))
    with result_out:
        print(f"saved {path}" + ("  (Colab: this file is in the temporary clone; download it from the Files panel)"
                                 if IN_COLAB else ""))
        print(path.read_text())


train_btn.on_click(on_train)
reset_btn.on_click(lambda _: set_widgets(BASELINE))
save_btn.on_click(on_save)
def on_start(change):
    try:
        set_widgets(yaml.safe_load(open(change["new"])))
    except ValueError as err:
        status.value = f"<span style='color:#c0392b'>{change['new']}: {err}</span>"


def refresh_run_lists():  # replaced in section 4 (Compare)
    pass


w["start"].observe(on_start, names="value")
for widget in w.values():
    widget.observe(_refresh_enabled, names="value")

tabs = W.Tab(children=[
    W.VBox([w["scaler"], w["features"], w["sampler"]]),
    W.VBox([w["hidden"], w["activation"], w["dropout"]]),
    W.VBox([W.HBox([W.VBox([w["optimizer"], w["lr"], w["momentum"], w["weight_decay"]]),
                    W.VBox([w["batch_size"], w["epochs"], w["early_stopping"], w["class_weight"]])]),
            w["seed"]]),
    W.VBox([w["rule"], w["threshold"], w["fpr"]]),
])
for i, title in enumerate(["Data", "Model", "Training", "Decision"]):
    tabs.set_title(i, title)

set_widgets(BASELINE)
display(W.VBox([
    W.HBox([w["start"], w["name"]]), tabs,
    W.HBox([train_btn, reset_btn, save_btn, live_box]), preview, status,
    live_out, result_out, table_out,
]))

## 4. Compare runs

Pick up to 4 runs from this session (the first one in the list is the reference, drawn in grey).
Same chart as `scripts/compare.py`. Keep the seed the same, otherwise the test sets differ.

In [ ]:
pick = W.SelectMultiple(description="runs", rows=8, layout=W.Layout(width="720px"))
compare_title = W.Text(value="Comparison", description="title", layout=W.Layout(width="420px"))
compare_btn = W.Button(description="Compare", icon="bar-chart")
compare_out = W.Output()


def refresh_run_lists():
    pick.options = list(RUNS)


def on_compare(_):
    with compare_out:
        clear_output(wait=True)
        names = list(pick.value)
        if not 1 <= len(names) <= 4:
            print("select 1 to 4 runs (Ctrl/Cmd-click to select several)")
            return
        compare(names, compare_title.value)
        display(results_table().loc[names])


compare_btn.on_click(on_compare)
refresh_run_lists()
display(W.VBox([pick, W.HBox([compare_title, compare_btn]), compare_out]))

## 5. Sweep: change one setting, keep everything else

Starts from the **current playground settings**, changes only the setting you pick, trains once
per value (up to 4 values, separated by `;`), then draws the comparison.
For `optimizer`, `sgd+momentum` means sgd with momentum 0.9.

In [ ]:
SWEEP_EXAMPLES = {
    "lr": "0.0001; 0.001; 0.01; 0.1",
    "optimizer": "sgd; sgd+momentum; rmsprop; adam",
    "batch_size": "32; 256; 4096",
    "hidden": "8; 64; 128, 64",
    "activation": "relu; tanh; sigmoid; leaky_relu",
    "dropout": "0; 0.2; 0.5",
    "weight_decay": "0; 0.0001; 0.001; 0.01",
    "scaler": "none; standard; minmax; log_standard",
    "class_weight": "none; balanced",
}
sweep_param = W.Dropdown(options=list(SWEEP_EXAMPLES), description="setting")
sweep_values = W.Text(value=SWEEP_EXAMPLES["lr"], description="values", layout=W.Layout(width="520px"))
sweep_btn = W.Button(description="Run sweep", button_style="warning", icon="play")
sweep_status, sweep_live, sweep_out = W.HTML(), W.Output(), W.Output()
sweep_param.observe(lambda ch: setattr(sweep_values, "value", SWEEP_EXAMPLES[ch["new"]]), names="value")


def sweep_cfg(base, param, text):
    cfg = copy.deepcopy(base)
    text = text.strip()
    if param == "optimizer":
        cfg["train"].pop("momentum", None)
        cfg["train"]["optimizer"] = "sgd" if text == "sgd+momentum" else text
        if text == "sgd+momentum":
            cfg["train"]["momentum"] = 0.9
    elif param in ("lr", "weight_decay", "dropout"):
        section = "model" if param == "dropout" else "train"
        cfg[section][param] = float(text)
        if param == "weight_decay" and cfg["train"]["weight_decay"] == 0:
            cfg["train"].pop("weight_decay")
    elif param == "batch_size":
        cfg["train"]["batch_size"] = int(text)
    elif param == "hidden":
        cfg["model"]["hidden"] = parse_hidden(text)
    elif param == "activation":
        cfg["model"]["activation"] = text
    elif param == "scaler":
        cfg["data"]["scaler"] = text
    elif param == "class_weight":
        cfg["train"]["class_weight"] = text
    cfg["name"] = auto_name(cfg)
    return cfg


def on_sweep(_):
    values = sweep_values.value.split(";")
    if not 1 <= len(values) <= 4:
        sweep_status.value = "give 1 to 4 values separated by ;"
        return
    base = current_cfg()
    sweep_btn.disabled = True
    sweep_out.clear_output()
    names = []
    try:
        for i, value in enumerate(values, 1):
            cfg = sweep_cfg(base, sweep_param.value, value)
            sweep_status.value = f"run {i}/{len(values)}: <b>{cfg['name']}</b> …"
            names.append(run(cfg, sweep_live if live_box.value else None)["name"])
    except KeyboardInterrupt:
        sweep_status.value = "<b>stopped.</b>"
    finally:
        sweep_btn.disabled = False
    if names:
        sweep_status.value = f"done: {len(names)} runs"
        with sweep_out:
            compare(names, f"Sweep: {sweep_param.value}")
            display(results_table().loc[names])
        refresh_run_lists()


sweep_btn.on_click(on_sweep)
display(W.VBox([W.HBox([sweep_param, sweep_values, sweep_btn]), sweep_status, sweep_live, sweep_out]))

## 6. Guided experiments

For each one: **guess first** (先猜再跑), then run it, then compare with what we saw.
"We saw" = one run on a laptop CPU, seed 42 (2026-10-18). On another machine or another
torch version the last digits can differ. Small differences between runs can be random.
Numbers are *validation loss* (lowest) and *test false alarms / attacks detected*.
The baseline is: Adam, lr 0.001, hidden `64`, standard scaling, early stopping 5, giving
**val loss 0.0083, false alarms 1.11%, attacks detected 99.79%**.

### A. Learning rate: how big is one step? (学习率)

*Sweep* → `lr` → `0.0001; 0.001; 0.01; 0.1`. Then try `1e-05` and `1.0` in the playground.

| lr | we saw |
| --- | --- |
| 0.00001 | still learning after 30 epochs: val loss 0.11, false alarms 35% |
| 0.0001 | val loss 0.028, false alarms 5.7% |
| 0.001 (baseline) | val loss 0.0083, false alarms 1.1% |
| 0.01 | val loss 0.0105, early stopping at epoch 18 |
| 0.1 | val loss jumps around (0.047 at epoch 4, 0.31 later), false alarms 12% |
| 1.0 | first-epoch loss 5.8: the steps overshoot |

Look at: the slope of the curves in the first few epochs, and how smooth the val curve is.

### B. Optimizer: how is the gradient used? (优化器)

*Sweep* → `optimizer` → `sgd; sgd+momentum; rmsprop; adam` (all at lr 0.001).

We saw: sgd val loss 0.0996 and 35% false alarms (too slow); sgd+momentum 0.047 / 6.7%;
rmsprop 0.0095 / 0.72%; adam 0.0083 / 1.1%.
Now set optimizer `sgd`, lr `0.1` in the playground: 0.016 / 2.1%.

Take-away: the best learning rate **depends on the optimizer**. Plain SGD moves by
lr × gradient, so it needs a bigger lr. Adam scales each weight's step by its recent gradient
size, so lr 0.001 works out of the box. Momentum 0.9 adds up past steps (roughly 10× bigger
steps when the gradient keeps pointing the same way).

### C. Batch size: how many rows per step? (批大小)

*Sweep* → `batch_size` → `32; 256; 4096`. Steps per epoch = 82,545 training rows ÷ batch size:
about 2,580 / 323 / 21.

We saw: batch 32 was about 4× slower per epoch (13 s for 16 epochs, baseline 6 s for 27) and
stopped at epoch 16 (val loss 0.0085);
batch 4096 made only 21 steps per epoch and after 30 epochs was still at val loss 0.033,
false alarms 5.7%.
Question: is "one epoch" a fair unit to compare them? What about "number of steps"?

### D. Scaling: why standardise? (标准化)

*Sweep* → `scaler` → `none; standard; minmax; log_standard`.

We saw: `none` starts with a training loss of about 44,000, keeps epoch 1 and detects only
97.5% of attacks. Some features reach ~2e10, so one weight × that feature dominates everything.
`minmax` 0.018, `standard` 0.0083, `log_standard` **0.0056** (best of the four; it first squashes
huge values with a log, then standardises).

### E. Network size (网络大小)

*Sweep* → `hidden` → `8; 64; 128, 64`, and try an **empty** hidden field in the playground:
no hidden layer = logistic regression (a single neuron).

We saw for the empty one: val loss 0.055, false alarms 7.5%. How much does the hidden layer buy?
Does a bigger network keep helping?

### F. Overfitting and how to fix it (过拟合与正则化, week 2 lesson 2)

With all 82,545 training rows this MLP hardly overfits (even `256, 128` for 100 epochs without
early stopping: best val loss 0.0058 at epoch 89, 0.0065 at the end). So make it overfit on purpose:

> playground: *training rows* `1% (subsample)` (828 rows), *hidden layers* `256, 128`,
> *max epochs* `150`, *early stopping* `0`

We saw: train loss falls to 0.0025, but **val loss is lowest at epoch 20 (0.066) and then climbs to
0.25**. The gap between the two curves is overfitting: the network learns the 828 rows by heart.

Now change **one** thing at a time and look at the val loss at the end:

| fix | we saw (val loss at the end) |
| --- | --- |
| nothing (above) | 0.25 |
| early stopping `5` | stops at epoch 25, keeps the epoch-20 weights (0.066) |
| weight decay `0.01` (adam) | **0.077** |
| weight decay `0.001` (adam) | 0.085 |
| smaller network: hidden `16` | 0.084 |
| dropout `0.3` / `0.5` | 0.26 / 0.23: barely helps here |
| scaler `log_standard` | lowest val loss 0.024 at epoch 47, 0.045 at the end |

Note: with `adamw` the weight decay is applied separately from the gradient, scaled by lr, so it
needs much bigger values to have an effect (we saw 0.17 at weight decay 1.0).
Discuss: why might dropout help so little in this setup?

### G. Deep network with sigmoid (梯度消失)

> hidden `64, 64, 64, 64`, activation `sigmoid`, optimizer `sgd`, lr `0.01`

We saw: the loss stays at about 0.33 from start to end and the model calls **everything** an
attack (false alarms 100%). The sigmoid's gradient is at most 0.25, and it is multiplied once per
layer on the way back, so the first layers hardly learn.
Same network with `relu`: 0.025 / 3.4%; with `tanh`: 0.030 / 3.2%; sigmoid with `adam` lr 0.001: 0.010 / 2.1%.

### H. False alarms vs missed attacks (类别不平衡与阈值)

- *class weight* `balanced` (retrains): false alarms 0.28%, attacks detected 99.52%.
- *Decision* → `max_fpr`, `0.005` (no change to the model, only the threshold, chosen on val):
  false alarms 0.50%, attacks detected 99.55%.

Compare with the baseline (1.11% / 99.79%). Which mistake costs more for an IoT network?

## 7. From the notebook to the report

- **Save as config file** (playground) writes `configs/notebook/<name>.yaml`. Run it with
  `python scripts/train.py configs/notebook/<name>.yaml` to get the full `runs/<name>/` folder
  (`summary.png`, `metrics.json`, …) for the slides. Same seed → the same numbers you saw here.
- In Colab the project is a temporary copy in `/content/net-attack-predict`: download files you
  want to keep from the Files panel on the left, or put the setting into a config in the repo.
- All results of this session as a table:

In [ ]:
results_table()